# Caculate F1 score

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics.classification import F1Score, BinaryF1Score
from sklearn.metrics import mean_absolute_error, accuracy_score
import numpy as np
import os
import pickle
import sys
sys.path.append("../../../")
from ctf_datasets.adni.dataset_SD import bin_array,ordinal_array



# === Main Evaluation Function ===
def evaluate_by_batch(loaded_results, batch_size=None, num_slices=10):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = torch.tensor(loaded_results['predictions'][attr]).squeeze()
        targets = torch.tensor(loaded_results['targets'][attr]).squeeze()
        
        if attr in ["pendulum","light","shadow_length", "shadow_position"]:
            metric = nn.L1Loss()
        else:
            continue  # Skip unknown attr

        
        # Regression
        preds_val = preds.cpu().numpy()
        targets_val = targets.cpu().numpy()
        if batch_size is None:
            l1 = mean_absolute_error(targets_val, preds_val)
            results_summary[attr] = {'L1-loss': round(l1, 3)}
        else:
            l1_list = []
            for i in range(0, len(preds_val), batch_size):
                p = preds_val[i:i+batch_size]
                t = targets_val[i:i+batch_size]
                l1 = mean_absolute_error(t, p)
                l1_list.append(l1)
            results_summary[attr] = {
                'L1-loss': round(np.mean(l1_list), 3),
                'L1-std': round(np.std(l1_list), 2)
            }

    return results_summary


In [3]:
# Example usage
# attr_keys = ['apoE', 'age', 'sex', 'brain_vol', 'vent_vol', 'slice']

target_attr = 'shadow_position'
result_path = "../saved/pendulum/effectiveness_step50.0_scale1.0/{}/final_results.pkl".format(target_attr)
with open(result_path, "rb") as f:
    loaded_results = pickle.load(f)

summary = evaluate_by_batch(loaded_results, batch_size=None, num_slices=10)
for key in summary.keys():
    print("{}:{}".format(key,summary[key]))


pendulum:{'L1-loss': 0.259}
light:{'L1-loss': 0.122}
shadow_length:{'L1-loss': 0.119}
shadow_position:{'L1-loss': 0.339}


In [4]:
nn.L1Loss()(torch.from_numpy(loaded_results['predictions']['pendulum']),torch.from_numpy(loaded_results['targets']['pendulum']))

tensor(0.2589)

In [5]:
# === Evaluation configuration ===

batch_size = None

# === Root directory of results ===
root_dir = "../saved_benchmark/pendulum/effectiveness_step50.0_scale1.0_blendFalse_caumechanism0"
#root_dir = "../saved_test_discovered/pendulum/effectiveness_step100.0_scale1.0_NTIFalse"
attr_keys = ["pendulum","light","shadow_length", "shadow_position"]

# === Find all subdirectories matching do(attr) ===
do_folders = [name for name in attr_keys if os.path.isdir(os.path.join(root_dir, name))]


for target_attr in ["pendulum","light","shadow_length", "shadow_position"]:
    print(f"Comparing metrics on attribute '{target_attr}' under different do(.) conditions with batch size = {batch_size}:\n")
    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue  # Skip if file missing

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        results = evaluate_by_batch(loaded_results, batch_size=batch_size)

        if target_attr not in results:
            continue  # Skip if attribute not found

        scores = results[target_attr]
        
        print(f"do({do_attr}): L1-loss = {scores['L1-loss']:3f} ± {scores.get('L1-std', 0):3f}")

Comparing metrics on attribute 'pendulum' under different do(.) conditions with batch size = None:

do(pendulum): L1-loss = 0.159000 ± 0.000000
do(light): L1-loss = 0.183000 ± 0.000000
do(shadow_length): L1-loss = 0.043000 ± 0.000000
do(shadow_position): L1-loss = 0.259000 ± 0.000000
Comparing metrics on attribute 'light' under different do(.) conditions with batch size = None:

do(pendulum): L1-loss = 0.060000 ± 0.000000
do(light): L1-loss = 0.137000 ± 0.000000
do(shadow_length): L1-loss = 0.058000 ± 0.000000
do(shadow_position): L1-loss = 0.120000 ± 0.000000
Comparing metrics on attribute 'shadow_length' under different do(.) conditions with batch size = None:

do(pendulum): L1-loss = 0.143000 ± 0.000000
do(light): L1-loss = 0.235000 ± 0.000000
do(shadow_length): L1-loss = 0.495000 ± 0.000000
do(shadow_position): L1-loss = 0.110000 ± 0.000000
Comparing metrics on attribute 'shadow_position' under different do(.) conditions with batch size = None:

do(pendulum): L1-loss = 0.086000 ± 0

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np
import os

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()

        binary_preds = (preds >= 0.5).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1, 3),
                'Accuracy': round(acc, 3)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 3),
                'F1-std': round(f1_std, 2),
                'Accuracy': round(acc_mean, 3),
                'Acc-std': round(acc_std, 2)
            }

    return results_summary


# === 设置评估目标属性 ===
target_attr = "Bald"
batch_size = 256

# === 根目录 ===
root_dir = "../saved/adni"
attr_keys = ['apoE', 'age', 'sex', 'brain_vol', 'vent_vol', 'slice']
# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in attr_keys if os.path.isdir(os.path.join(root_dir, name))]

print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)

    results = evaluate_by_batch(loaded_results, batch_size=batch_size)

    if target_attr not in results:
        continue  # skip if target attr not in result

    scores = results[target_attr]
    if batch_size is None:
        print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
    else:
        print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")


Comparing '256' F1-score on attribute 'Bald' under different do(.) conditions:
do(Young)	Bald: F1 = 0.941 ± 0.02, Acc = 0.973 ± 0.01
do(Male)	Bald: F1 = 0.944 ± 0.04, Acc = 0.99 ± 0.01
do(No_Beard)	Bald: F1 = 0.891 ± 0.11, Acc = 0.995 ± 0.0
do(Bald)	Bald: F1 = 0.494 ± 0.04, Acc = 0.342 ± 0.03


In [ ]:
#dict_keys(['predictions', 'targets'])
loaded_results.keys()
#dict_keys(['Young', 'Male', 'No_Beard', 'Bald'])
loaded_results['predictions'].keys()

dict_keys(['predictions', 'targets'])

# FID and Minimality

In [1]:
import torch
minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
countefactual_images =  torch.load("./saved/celeA_complex/FID/counterfactual_tensors.pt")

/tmp/ipykernel_290807/4134258590.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
/tmp/ipykernel_290807

In [6]:
minimality['factuals'][1].shape

AttributeError: 'tuple' object has no attribute 'shape'